# Phase 4 - Court Judgment Prediction (CJPE) - Architecture 3 (linear-attention / recurrent hybrid)

**Task:** binary court-judgment-outcome prediction (labels 0 / 1) from the full text of a case. Primary metric: **macro-F1**.
**Phase-1 model:** Architecture 3 (linear-attention / recurrent hybrid) - Gated linear-attention layers (input-dependent decay, short causal conv, L2-normalised q/k) with a softmax-attention layer every 4th layer (baseline: RetNet-style retention in every layer). `g` is the output gate of the linear-attention layer.
**Adaptation (this notebook):** **bottleneck adapters**, one `METHOD = ...` switch away from the other Assignment-4 techniques (LoRMA, LoRA, adapters, soft prompts) and full fine-tuning.
**No knowledge distillation** in this notebook (distillation is done in `Phase4_Arch_1_LoRMA_KD.ipynb`).

**How a prediction is made.** A judgment is cut into overlapping **chunks** of `CHUNK_LEN` tokens (stride `CHUNK_STRIDE`); very long cases keep their head and their tail. The Phase-1 model reads every chunk - causal hybrid: the chunk is processed in parallel (chunkwise linear attention); the **mean** of the token states and the state of the **last** token are the chunk vector. A **hierarchical document head** adds an embedding of the chunk's relative position in the judgment (section-aware), pools the chunk vectors (`DOC_POOL`: attention / mean / BiGRU / last chunk) and a small MLP gives the outcome. `SECTION_MODE` selects which part of the judgment is read (`full`, `last`, `first`, `head_tail`).

| step | what happens |
|---|---|
| 1-3 | setup, configuration, your Phase-1 tokenizer (unchanged) |
| 4-5 | load CJPE data, tokenise, class balance, chunking and batching |
| 6 | macro-F1 / precision / recall, threshold tuning, calibration, false-positive / false-negative analysis |
| 7-9 | your Phase-1 model (unchanged), adaptation methods of Assignment 4, document classifier |
| 10-12 | training code, then run: train, validate, section analysis, predict test, write `submission.zip` |
| 13 | save trainable weights and `results.json` |

**Before running:** attach your Phase-1 checkpoint (`best.pt`) and `kratikasum/tokenizer` as Kaggle inputs and keep the secret `HF`. Run once with `SMOKE_TEST = True`, then switch it off. Only Phase-1 weights and the provided CJPE data are used.

**Outputs** (in `/kaggle/working/<RUN_NAME>/`): `prediction.json` (`[{"id": ..., "pred": 0/1}, ...]`), `submission.zip`, `results.json`, `dev_probs.npy` / `test_probs.npy` (for ensembling), the trained weights.

## 1. Setup

### Imports, Hugging Face login, GPU check

In [144]:
import os, re, sys, json, glob, math, time, random, zipfile, gc, pickle, copy
from collections import Counter
from dataclasses import dataclass, asdict, fields
from typing import List, Dict, Tuple, Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import login, hf_hub_download
from tqdm.auto import tqdm

try:                                                     # Kaggle secret 'HF' (same as in your Phase-1 / 2 / 3 notebooks)
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
except Exception as ex:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    print(f"Kaggle secret 'HF' not available ({type(ex).__name__}); using env HF_TOKEN / local files")

print('CUDA available:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))
N_GPU = torch.cuda.device_count()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_bf16 = device == 'cuda' and torch.cuda.get_device_capability()[0] >= 8      # T4 / P100: fp16 + GradScaler, Ampere+: bf16
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


CUDA available: True
GPU 0: Tesla T4
GPU 1: Tesla T4


## 2. Configuration
Every setting of the experiment is in this section.

### Paths and run name

In [145]:
# ---------- where things are ----------
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
PHASE4_DIR      = 'Phase_4_CJPE'        # Train/train.json, Train/dev.json, Test/test.json inside that dataset
LOCAL_DATA_DIR  = None                  # or e.g. '/kaggle/input/<dataset>/Phase_4_CJPE'
PHASE1_CKPT     = ''                    # your Phase-1 best.pt / last.pt   ('' = search /kaggle/input)
CKPT_HINT       = 'arch3'       # used by that search: the path must contain this text ('-' and '_' are ignored)
TOKENIZER_PKL   = ''                    # '' = search /kaggle/input, then download kratikasum/tokenizer
RUN_NAME        = 'p4_arch3_adapter'
OUT_ROOT        = '/kaggle/working' if os.path.isdir('/kaggle/working') else './outputs'


### Long-document representation, pooling and class balance

In [146]:
# ---------- how a long judgment is represented ----------
SEED            = 42
CHUNK_LEN       = 1024                  # tokens per chunk: the Phase-1 model reads one chunk at a time
CHUNK_STRIDE    = 896                   # < CHUNK_LEN -> neighbouring chunks overlap by CHUNK_LEN - CHUNK_STRIDE tokens
MAX_DOC_TOKENS  = 12288                 # a longer judgment keeps its first HEAD_FRAC and its last (1 - HEAD_FRAC) share of tokens
HEAD_FRAC       = 0.25
SECTION_MODE    = 'full'                # which part of the judgment is read: 'full' | 'last' | 'first' | 'head_tail'
TRAIN_CHUNKS    = 6                     # chunks per document in one training step (random subset, document order is kept)
TRAIN_KEEP_LAST = True                  # the last chunk (the court's closing reasoning) is always one of them
MAX_EVAL_CHUNKS = 24                    # chunks per document at validation / test time (evenly spread, first + last included)
DOC_POOL        = 'attn'                # chunk vectors -> document vector: 'attn' | 'mean' | 'gru' | 'last'
HEAD_HIDDEN     = 256                   # width of the document-level head
N_POS_BINS      = 8                     # the relative position of a chunk inside the judgment is embedded (section-aware head)
CW_POWER        = 0.5                   # class weight = (N / (C * count_label)) ** CW_POWER   (0 = off)
LABEL_SMOOTH    = 0.05


### Adaptation method

In [147]:
# ---------- adaptation method (the four techniques of Assignment 4, plus full fine-tuning as a reference) ----------
METHOD          = 'adapter'            # 'lorma' | 'lora' | 'adapter' | 'soft_prompt' | 'full'   (see section 8)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05              # 'lora' / 'lorma': rank, alpha (scale = alpha / r = 2, as in Assignment 4), dropout
PEFT_TARGETS    = ('q', 'k', 'v', 'g', 'o', 'gate', 'up', 'down')   # linear layers that receive LoRA / LoRMA matrices
ADAPTER_BOTTLENECK = 64                 # 'adapter': bottleneck width of the residual adapter after every block
N_PROMPT        = 32                    # 'soft_prompt': number of learned input vectors
PROMPT_INIT_TEXT = ' court judgment prediction: read the case and decide whether the appeal is accepted or rejected:'
BACKBONE_DROPOUT = 0.1                  # dropout inside the Phase-1 blocks while fine-tuning


### Training

In [148]:
# ---------- training ----------
EPOCHS          = 2
LR_BY_METHOD    = dict(lorma=3e-4, lora=3e-4, adapter=5e-4, soft_prompt=3e-3, full=3e-5)   # learning rate of the adapted backbone parameters
HEAD_LR         = 1e-3                  # document head (always trained from scratch)
BATCH_DOCS, GRAD_ACCUM = 2, 4           # documents per forward pass x accumulation  (= 8 documents per optimizer step)
EPOCH_DOCS      = 12000                  # None = every training document each epoch, or e.g. 8000 to subsample (faster epochs)
GRAD_CKPT       = True                  # saves GPU memory (~30% slower). Set False if you have memory to spare
EVAL_EVERY      = 250                   # optimizer steps between two quick validations
QUICK_DEV_DOCS, QUICK_EVAL_CHUNKS = 800, 8    # size of the quick validation (the full dev set is used at the end)
EVAL_CHUNKS_PER_BATCH = 8               # chunks per forward pass at inference time
LOG_EVERY       = 20
PATIENCE        = 10                     # stop after this many quick validations without improvement
MAX_TRAIN_MINUTES = 300                 # wall-clock cap: the LR schedule is stretched so that it always finishes decaying
APPLY_TUNED_THRESHOLD = True            # test predictions use the decision threshold that maximises dev macro-F1 (else 0.5)


### Quick dry run, seeds and output folder

In [149]:
# ---------- quick dry run ----------
SMOKE_TEST = False                      # True = a handful of documents and 6 steps: checks the whole pipeline in minutes
MAX_DOCS = MAX_STEPS = None
if SMOKE_TEST:
    MAX_DOCS, MAX_STEPS = 300, 60
    RUN_NAME += '_smoke'
    EPOCHS, EVAL_EVERY, QUICK_DEV_DOCS, MAX_EVAL_CHUNKS = 1, 3, 12, 6

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
LR = LR_BY_METHOD[METHOD]
OUTPUT_DIR = os.path.join(OUT_ROOT, RUN_NAME)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('run:', RUN_NAME, '| method:', METHOD, '| output folder:', OUTPUT_DIR)

run: p4_arch3_adapter | method: adapter | output folder: /kaggle/working/p4_arch3_adapter


## 3. Your Phase-1 tokenizer
The class is your Phase-1 code, unchanged.

### Tokenizer class

In [150]:
class ByteLevelBPE:
    """Your tokenizer class (training code removed: we load the already-trained pickle)."""
    def __init__(self):
        self.vocab: Dict[bytes, int] = {}
        self.inverse_vocab: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], Tuple[bytes, int]] = {}
        self.word_end_marker = b'</w>'
        self.reserved_tokens = [b'<pad>', b'<unk>', b'<bos>', b'<eos>']

    def tokenize(self, text: str) -> List[bytes]:
        words = text.split()
        tokens = []
        for w in words:
            word_bytes = w.encode('utf-8')
            split = [bytes([b]) for b in word_bytes] + [self.word_end_marker]
            while True:
                best_pair, best_rank = None, float('inf')
                for i in range(len(split) - 1):
                    pair = (split[i], split[i+1])
                    if pair in self.merges:
                        rank = self.merges[pair][1]
                        if rank < best_rank:
                            best_rank, best_pair = rank, pair
                if best_pair is None:
                    break
                new_split, i = [], 0
                while i < len(split):
                    if i < len(split) - 1 and split[i] == best_pair[0] and split[i+1] == best_pair[1]:
                        new_split.append(self.merges[best_pair][0])
                        i += 2
                    else:
                        new_split.append(split[i])
                        i += 1
                split = new_split
            tokens.extend(split)
        return tokens

    def encode(self, text: str) -> List[int]:
        unk_id = self.vocab[b'<unk>']
        return [self.vocab.get(token, unk_id) for token in self.tokenize(text)]

    def decode(self, token_ids: List[int]) -> str:
        skip_ids = {self.vocab[t] for t in self.reserved_tokens if t in self.vocab}
        bytes_list = []
        for tid in token_ids:
            if tid in skip_ids:
                continue
            bytes_list.append(self.inverse_vocab.get(tid, b''))
        joined = b''.join(bytes_list)
        return joined.replace(self.word_end_marker, b' ').decode('utf-8', errors='replace').strip()

    @classmethod
    def load(cls, filepath: str):
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        bpe = cls()
        bpe.vocab = data['vocab']
        bpe.inverse_vocab = {v: k for k, v in bpe.vocab.items()}
        bpe.merges = data['merges']
        return bpe

    def __len__(self):
        return len(self.vocab)

    @property
    def eos_token_id(self):
        return self.vocab.get(b'<eos>')

### Load the trained tokenizer, check vocabulary size

In [151]:
if TOKENIZER_PKL and os.path.exists(TOKENIZER_PKL):
    tok_path = TOKENIZER_PKL
else:
    cands = glob.glob('/kaggle/input/**/bpe_tokenized_txt.pkl', recursive=True)
    if cands:
        tok_path = cands[0]
    else:                                   # dataset not attached as input -> download it
        import kagglehub
        tok_path = os.path.join(kagglehub.dataset_download('kratikasum/tokenizer'), 'bpe_tokenized_txt.pkl')
tokenizer = ByteLevelBPE.load(tok_path)

VOCAB_SIZE = max(tokenizer.vocab.values()) + 1
PAD_ID, EOS = tokenizer.vocab[b'<pad>'], tokenizer.eos_token_id
print(f'tokenizer: {len(tokenizer):,} entries | embedding size {VOCAB_SIZE:,} | marker {tokenizer.word_end_marker}')

tokenizer: 48,000 entries | embedding size 48,000 | marker b'</w>'


### Quick encode / decode test

In [152]:
ids = tokenizer.encode("The Supreme Court dismissed the appeal filed by the petitioner.")
print(ids); print(tokenizer.decode(ids))

[350, 2591, 402, 1688, 265, 745, 836, 319, 265, 7419]
The Supreme Court dismissed the appeal filed by the petitioner.


## 4. Load the CJPE data
`train.json` / `dev.json` (id, text, label) and `test.json` (id, text). Every case is tokenised with your Phase-1 tokenizer and closed with `<eos>`.

### Helpers: file lookup, streaming JSON reader, chunk offsets

In [153]:
try:
    import ijson                            # streams the (large) JSON files record by record
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ijson'], check=False)
    try:
        import ijson
    except ImportError:
        ijson = None
        print('ijson not available -> the JSON files are read in one go')

word_cache = {}                             # the Phase-1 tokenizer works word by word, so caching per word is exact


def encode_words(words):
    out = []
    for w in words:
        ids = word_cache.get(w)
        if ids is None:
            ids = word_cache[w] = tokenizer.encode(w)
        out.extend(ids)
    return out


def find_data_file(name, sub):
    """Local override -> attached Kaggle dataset -> Hugging Face dataset repo."""
    if LOCAL_DATA_DIR:
        for p in (os.path.join(LOCAL_DATA_DIR, sub, name), os.path.join(LOCAL_DATA_DIR, name)):
            if os.path.exists(p):
                return p
    hits = [h for h in glob.glob(f'/kaggle/input/**/{name}', recursive=True) if PHASE4_DIR.lower() in h.lower() or 'cjpe' in h.lower()]
    if hits:
        return hits[0]
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=f'{PHASE4_DIR}/{sub}/{name}', repo_type='dataset', token=HF_TOKEN)


def iter_records(path):
    """Yields the case records of a JSON list (streamed with ijson) or of a {case_id: record} dictionary."""
    with open(path, 'rb') as f:
        first = f.read(1)
        while first and first.isspace():
            first = f.read(1)
    if first == b'[' and ijson is not None:
        with open(path, 'rb') as f:
            yield from ijson.items(f, 'item')
    else:
        raw = json.load(open(path, encoding='utf-8'))
        if isinstance(raw, dict):
            for k, v in raw.items():
                v = dict(v); v.setdefault('id', k)
                yield v
        else:
            yield from raw


def chunk_starts(n, chunk_len, stride):
    """Start offsets of the overlapping chunks of a document of n tokens; the last chunk is aligned with the end of the document."""
    if n <= chunk_len:
        return [0]
    starts = list(range(0, n - chunk_len + 1, stride))
    if starts[-1] + chunk_len < n:
        starts.append(n - chunk_len)
    return starts


def head_tail(seq, limit, head_frac):
    if len(seq) <= limit:
        return seq
    h = int(limit * head_frac)
    return seq[:h] + seq[len(seq) - (limit - h):]


def load_split(path, labeled, max_docs=None):
    """-> list of dict(id, tok (int32 array incl. <eos>), starts, n_words, y).  Very long cases keep their head and their tail."""
    docs = []
    for i, rec in enumerate(tqdm(iter_records(path), desc=os.path.basename(path), leave=False)):
        if max_docs and i >= max_docs:
            break
        words = str(rec.get('text', '')).split()
        ids = encode_words(head_tail(words, MAX_DOC_TOKENS, HEAD_FRAC))      # a word is >= 1 token, so this never loses a needed token
        ids = head_tail(ids, MAX_DOC_TOKENS - 1, HEAD_FRAC) + [EOS]
        tok = np.asarray(ids, np.int32)
        docs.append(dict(id=str(rec['id']), tok=tok, starts=chunk_starts(len(tok), CHUNK_LEN, CHUNK_STRIDE), n_words=len(words),
                         y=int(rec['label']) if labeled else -1))
    return docs

### Load train / dev / test

In [154]:
train_docs = load_split(find_data_file('train.json', 'Train'), True, MAX_DOCS)
dev_docs   = load_split(find_data_file('dev.json', 'Train'), True, MAX_DOCS)
test_docs  = load_split(find_data_file('test.json', 'Test'), False, MAX_DOCS)
assert len({d['id'] for d in test_docs}) == len(test_docs), 'duplicate case ids in test.json'
print(f'tokenizer cache: {len(word_cache):,} distinct words')

train.json: 0it [00:00, ?it/s]

dev.json: 0it [00:00, ?it/s]

test.json: 0it [00:00, ?it/s]

tokenizer cache: 787,862 distinct words


### Statistics

In [155]:
def describe(name, docs, labeled=True):
    n_tok = np.array([len(d['tok']) for d in docs]); n_ch = np.array([len(d['starts']) for d in docs])
    lab = f' | label counts {dict(sorted(Counter(d["y"] for d in docs).items()))}' if labeled else ''
    print(f'{name:5s}: {len(docs):6d} cases{lab}\n       tokens/case mean {n_tok.mean():7.0f} | median {np.median(n_tok):7.0f} | '
          f'90th pct {np.percentile(n_tok, 90):7.0f} | max {n_tok.max():6d} | chunks/case mean {n_ch.mean():5.1f} (max {n_ch.max()})')


describe('train', train_docs); describe('dev', dev_docs); describe('test', test_docs, labeled=False)

train:  37387 cases | label counts {0: 22067, 1: 15320}
       tokens/case mean    3269 | median    2443 | 90th pct    7333 | max  12288 | chunks/case mean   4.0 (max 14)
dev  :   1988 cases | label counts {0: 994, 1: 994}
       tokens/case mean    4086 | median    3301 | 90th pct    7964 | max  12288 | chunks/case mean   4.9 (max 14)
test :   1517 cases
       tokens/case mean    4195 | median    3387 | 90th pct    8304 | max  12288 | chunks/case mean   5.0 (max 14)


### Class balance and class weights
The loss is class-weighted (`CW_POWER`); the decision threshold is tuned for macro-F1 later (section 11).

In [156]:
NUM_CLASSES = 2
counts = np.bincount([d['y'] for d in train_docs], minlength=NUM_CLASSES).astype(np.float64)
cw = np.ones(NUM_CLASSES, np.float32)
if CW_POWER > 0:
    cw = ((counts.sum() / (NUM_CLASSES * np.maximum(counts, 1))) ** CW_POWER).astype(np.float32)
    cw = cw / cw[counts > 0].mean()                                                      # mean weight ~1
CLASS_W = torch.tensor(cw, device=device)
print('training cases per label / loss weight (class balancing):')
for c in range(NUM_CLASSES):
    print(f'   label {c}: {int(counts[c]):6d} ({100 * counts[c] / counts.sum():5.1f}%)   weight {cw[c]:.3f}')

training cases per label / loss weight (class balancing):
   label 0:  22067 ( 59.0%)   weight 0.909
   label 1:  15320 ( 41.0%)   weight 1.091


## 5. Chunks and batches
A judgment becomes a list of overlapping chunks. During training a random subset of `TRAIN_CHUNKS` chunks (document order kept, last chunk always included) is read; at validation / test time up to `MAX_EVAL_CHUNKS` evenly spread chunks are read.

### Chunk selection and batch builder

In [157]:
def select_chunks(n, mode, cap, rng=None, keep_last=True):
    """Indices (into the document's list of n chunks) that the model reads.
    mode: 'full' every chunk | 'first' | 'last' | 'head_tail' (first + last).  More than `cap` chunks are thinned out:
    random subset in document order while training (the last chunk is kept if keep_last), evenly spread when rng is None."""
    if mode == 'first':
        idx = [0]
    elif mode == 'last':
        idx = [n - 1]
    elif mode == 'head_tail':
        idx = sorted({0, n - 1})
    elif mode == 'full':
        idx = list(range(n))
    else:
        raise ValueError(f'unknown SECTION_MODE {mode!r}')
    if len(idx) > cap:
        if rng is not None:
            if keep_last:
                pick = sorted(rng.choice(np.array(idx[:-1]), cap - 1, replace=False).tolist()) + [idx[-1]]
            else:
                pick = sorted(rng.choice(np.array(idx), cap, replace=False).tolist())
            idx = pick
        else:
            idx = sorted({idx[int(round(i))] for i in np.linspace(0, len(idx) - 1, cap)})
    return idx


def make_batch(docs, doc_ids, train, rng=None, mode=None, max_chunks=None):
    """Documents -> one tensor of chunks.  ids/mask: (N_chunks, T) | doc_idx/slot: which document / which slot of it each chunk is |
    pos_bin: relative position of the chunk in its judgment | y: label of every document"""
    mode = mode or SECTION_MODE
    max_chunks = max_chunks or (TRAIN_CHUNKS if train else MAX_EVAL_CHUNKS)
    pieces, doc_idx, slot, pos_bin, rel = [], [], [], [], []
    for b, di in enumerate(doc_ids):
        d = docs[di]
        starts = d['starts']
        sel = select_chunks(len(starts), mode, max_chunks, rng if train else None, TRAIN_KEEP_LAST)
        rel.append(np.array([starts[i] / max(1, starts[-1]) for i in sel], np.float32))
        for j, i in enumerate(sel):
            pieces.append(d['tok'][starts[i]:starts[i] + CHUNK_LEN])
            doc_idx.append(b); slot.append(j)
            pos_bin.append(int(round(rel[-1][j] * (N_POS_BINS - 1))))
    T = max(len(p) for p in pieces)
    ids = np.full((len(pieces), T), PAD_ID, np.int64); mask = np.zeros((len(pieces), T), bool)
    for r, p in enumerate(pieces):
        ids[r, :len(p)], mask[r, :len(p)] = p, True
    to = lambda a: torch.from_numpy(a).to(device)
    return dict(ids=to(ids), mask=to(mask), doc_idx=to(np.array(doc_idx, np.int64)), slot=to(np.array(slot, np.int64)),
                pos_bin=to(np.array(pos_bin, np.int64)), n_docs=len(doc_ids), max_slots=int(max(slot)) + 1, rel=rel,
                y=to(np.array([docs[i]['y'] for i in doc_ids], np.int64)))

## 6. Metrics and error analysis
* **Macro-F1** (primary), macro precision and recall, per-label scores and the confusion matrix;
* decision-threshold tuning for macro-F1 and a calibration check (ECE);
* characteristics of **false positives / false negatives**: size, document length, confidence, most confident mistakes.

### Macro-F1 / precision / recall, threshold tuning, calibration

In [158]:
def binary_scores(y_true, y_pred):
    """Macro-F1 (primary), macro precision / recall, accuracy, confusion matrix cm[gold, predicted] and per-label scores."""
    cm = np.bincount(y_true * 2 + y_pred, minlength=4).reshape(2, 2)
    tp, sup, prd = np.diag(cm).astype(float), cm.sum(1), cm.sum(0)
    p, r = tp / np.maximum(prd, 1), tp / np.maximum(sup, 1)
    f = 2 * p * r / np.maximum(p + r, 1e-12)
    return dict(macro_f1=float(f.mean()), macro_p=float(p.mean()), macro_r=float(r.mean()), accuracy=float(tp.sum() / max(1, cm.sum())),
                cm=cm.tolist(), per_label={c: dict(p=float(p[c]), r=float(r[c]), f1=float(f[c]), support=int(sup[c])) for c in range(2)})


def tune_threshold(y_true, p1, grid=np.linspace(0.05, 0.95, 91)):
    """Decision threshold on P(label = 1) that maximises macro-F1 (post-processing for the class imbalance)."""
    best_t, best_f = 0.5, -1.0
    for t in grid:
        f = binary_scores(y_true, (p1 >= t).astype(int))['macro_f1']
        if f > best_f + 1e-12:
            best_t, best_f = float(t), f
    return best_t, best_f


def expected_calibration_error(y_true, probs, bins=10):
    conf, pred = probs.max(1), probs.argmax(1)
    ece, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs((pred[m] == y_true[m]).mean() - conf[m].mean())
    return float(ece)


def print_scores(s, title=''):
    print(f'--- {title} macro-F1 {s["macro_f1"]:.4f} | macro-P {s["macro_p"]:.4f} | macro-R {s["macro_r"]:.4f} | accuracy {s["accuracy"]:.4f}')
    for c in (0, 1):
        l = s['per_label'][c]
        print(f'      label {c}: precision {l["p"]:.4f} | recall {l["r"]:.4f} | F1 {l["f1"]:.4f} | support {l["support"]}')
    cm = np.array(s['cm'])
    print(f'      confusion (rows = gold, columns = predicted):  [[{cm[0, 0]:5d} {cm[0, 1]:5d}]  [{cm[1, 0]:5d} {cm[1, 1]:5d}]]')

### False-positive / false-negative analysis

In [159]:
def error_analysis(docs, probs, thr, top=5):
    """Characteristics of the four outcome groups (label 1 = positive class): size, document length, confidence, and the most confident mistakes."""
    y = np.array([d['y'] for d in docs]); p1 = probs[:, 1]; pred = (p1 >= thr).astype(int)
    n_tok = np.array([len(d['tok']) for d in docs]); n_ch = np.array([len(d['starts']) for d in docs])
    groups = {'TP': (y == 1) & (pred == 1), 'TN': (y == 0) & (pred == 0), 'FP': (y == 0) & (pred == 1), 'FN': (y == 1) & (pred == 0)}
    out = dict(groups={}, by_length={}, most_confident_errors={})
    print(f'{"group":6s}{"cases":>7s}{"tokens/case":>13s}{"chunks/case":>13s}{"mean P(1)":>11s}')
    for name, m in groups.items():
        out['groups'][name] = dict(count=int(m.sum()), mean_tokens=float(n_tok[m].mean()) if m.any() else None,
                                   mean_chunks=float(n_ch[m].mean()) if m.any() else None, mean_p1=float(p1[m].mean()) if m.any() else None)
        g = out['groups'][name]
        print(f'{name:6s}{g["count"]:7d}' + ''.join(f'{g[k]:{w}.{q}f}' if g[k] is not None else f'{"-":>{w}s}'
                                                       for k, w, q in (('mean_tokens', 13, 0), ('mean_chunks', 13, 1), ('mean_p1', 11, 3))))
    edges = np.quantile(n_tok, [0, .25, .5, .75, 1.0])
    print('error behaviour by document length (quartiles of tokens):')
    for q in range(4):
        m = (n_tok >= edges[q]) & ((n_tok <= edges[q + 1]) if q == 3 else (n_tok < edges[q + 1]))
        if m.sum() == 0:
            continue
        s = binary_scores(y[m], pred[m])
        out['by_length'][f'Q{q + 1}'] = dict(tokens=f'{int(edges[q])}-{int(edges[q + 1])}', cases=int(m.sum()), macro_f1=s['macro_f1'], accuracy=s['accuracy'])
        print(f'   Q{q + 1} ({int(edges[q]):6d}-{int(edges[q + 1]):6d} tokens, {int(m.sum()):4d} cases): accuracy {s["accuracy"]:.3f} | macro-F1 {s["macro_f1"]:.3f}')
    for name in ('FP', 'FN'):
        idx = np.where(groups[name])[0]
        conf = (p1 if name == 'FP' else 1 - p1)[idx]
        pick = idx[np.argsort(-conf)[:top]]
        out['most_confident_errors'][name] = [dict(id=docs[i]['id'], p1=float(p1[i]), tokens=int(n_tok[i])) for i in pick]
        print(f'most confident {name}:', [(docs[i]['id'], round(float(p1[i]), 3)) for i in pick])
    return out

## 7. Your Phase-1 model
The architecture code is copied **unchanged** from your Phase-1 notebook; the weights come from your checkpoint.

### Model code (hand-written, unchanged from Phase 1)

In [160]:
import math
from dataclasses import dataclass
from types import SimpleNamespace
from typing import List, Optional
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

### Config

In [161]:
@dataclass
class Config:
    vocab_size: int = 32000
    d_model: int = 768
    n_layer: int = 12
    n_head: int = 12
    n_kv_head: Optional[int] = None
    max_seq_len: int = 2048
    rope_theta: float = 10000.0
    rope_pct: float = 1.0
    dropout: float = 0.0
    tie_embeddings: bool = False
    qk_norm: bool = False
    grad_ckpt: bool = False
    layer_types: Optional[List[str]] = None   # per layer: "r" linear attention, "n" global softmax attention (NoPE)
    la_decay: str = "fixed"              # "fixed" (RetNet) | "data" (GLA-style, input dependent)
    la_conv: int = 0                     # short causal depthwise conv kernel (0 = off)
    la_l2norm: bool = False
    la_rope: bool = True
    la_chunk: int = 64

    @property
    def head_dim(self):
        return self.d_model // self.n_head

    @property
    def kv_heads(self):
        return self.n_kv_head or self.n_head

    @property
    def ff_dim(self):
        return int(round(8 / 3 * self.d_model / 64)) * 64

### Building blocks: RMSNorm, RoPE, SwiGLU MLP

In [162]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        out = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (out * self.w.float()).type_as(x)


class Rotary(nn.Module):
    def __init__(self, dim, max_len, theta):
        super().__init__()
        inv = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / max(dim, 1)))
        freqs = torch.outer(torch.arange(max_len).float(), inv)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, T, start=0):
        """cos/sin for positions start .. start+T-1 (start > 0 only while generating with a cache)."""
        return self.cos[start:start + T], self.sin[start:start + T]


def apply_rope(x, cos, sin, rot_dim):
    """x: (B,H,T,hd). Rotates the first rot_dim dims (half-split convention)."""
    if rot_dim == 0:
        return x
    xr, xp = x[..., :rot_dim], x[..., rot_dim:]
    x1, x2 = xr.float().chunk(2, dim=-1)
    c, s = cos[None, None], sin[None, None]
    xr = torch.cat([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).to(x.dtype)
    return torch.cat([xr, xp], dim=-1)


class MLP(nn.Module):
    """SwiGLU feed-forward."""
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg.d_model, cfg.ff_dim
        self.gate = nn.Linear(d, f, bias=False)
        self.up = nn.Linear(d, f, bias=False)
        self.down = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

### KV cache (new: lets generation process one token per step instead of the whole prefix)

In [163]:
# --------------------------------------------------------------------------------------
# KV cache + attention
# --------------------------------------------------------------------------------------
class KVCache:
    """Pre-allocated key/value buffer for ONE attention layer (stored un-expanded, i.e. Hkv heads).
    Lets generation process one new token per step instead of re-running the whole prefix."""

    def __init__(self, max_len):
        self.max_len, self.n, self.k, self.v = max_len, 0, None, None

    def append(self, k, v):
        if self.k is None:
            B, H, _, hd = k.shape
            self.k = k.new_empty(B, H, self.max_len, hd)
            self.v = v.new_empty(B, H, self.max_len, hd)
        T = k.size(2)
        self.k[:, :, self.n:self.n + T] = k
        self.v[:, :, self.n:self.n + T] = v
        self.n += T

    def get(self, window=None):
        s = 0 if window is None else max(0, self.n - window)
        return self.k[:, :, s:self.n], self.v[:, :, s:self.n]


def sdpa_gqa(q, k, v, mask=None, causal=False, p=0.0):
    """q: (B,H,T,hd)   k,v: (B,Hkv,S,hd).  Handles GQA without copying the cache when T == 1."""
    B, H, T, hd = q.shape
    Hkv = k.size(1)
    if T == 1 and mask is None:                       # single-token decode step
        if Hkv != H:                                  # fold the query-head group into the "length" axis
            r = H // Hkv
            out = F.scaled_dot_product_attention(q.reshape(B, Hkv, r, hd), k, v)
            return out.reshape(B, H, 1, hd)
        return F.scaled_dot_product_attention(q, k, v)
    if Hkv != H:
        r = H // Hkv
        k, v = k.repeat_interleave(r, dim=1), v.repeat_interleave(r, dim=1)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=p,
                                          is_causal=causal and mask is None)

### Self-attention (GQA / QK-norm / RoPE-or-NoPE / optional value residual, with KV-cache support)

In [164]:
class Attention(nn.Module):
    """MHA/GQA self-attention with optional RoPE (or NoPE), QK-norm, value residual, sliding-window mask
    and KV cache.  Returns (output, v_of_the_new_tokens)."""

    def __init__(self, cfg, causal=True, use_rope=True, value_residual=False):
        super().__init__()
        d = cfg.d_model
        self.H, self.Hkv, self.hd = cfg.n_head, cfg.kv_heads, cfg.head_dim
        assert self.H % self.Hkv == 0, "n_head must be divisible by n_kv_head"
        self.q = nn.Linear(d, self.H * self.hd, bias=False)
        self.k = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.v = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.o = nn.Linear(self.H * self.hd, d, bias=False)
        self.q_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.k_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.p_drop = cfg.dropout
        self.causal = causal
        self.rot_dim = (int(self.hd * cfg.rope_pct) // 2 * 2) if use_rope else 0
        self.lam = nn.Parameter(torch.zeros(self.Hkv, 1, 1)) if value_residual else None

    def forward(self, x, rope=None, v_first=None, mask=None, cache=None, window=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.H, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)
        if self.lam is not None and v_first is not None:
            lam = torch.sigmoid(self.lam).to(v.dtype)
            v = (1 - lam) * v + lam * v_first
        v_new = v
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        if self.rot_dim:
            cos, sin = rope
            q, k = apply_rope(q, cos, sin, self.rot_dim), apply_rope(k, cos, sin, self.rot_dim)
        causal = self.causal
        if cache is not None:
            cache.append(k, v)
            k, v = cache.get(window if T == 1 else None)   # window only matters for 1-token steps
            causal = T > 1                                  # prefill: causal; decode step: sees everything cached
        out = sdpa_gqa(q, k, v, mask=mask, causal=causal, p=self.p_drop if self.training else 0.0)
        return self.o(out.transpose(1, 2).reshape(B, T, -1)), v_new

### LM head (final norm + tied unembedding + multi-token-prediction heads)

In [165]:
class LMHead(nn.Module):
    """Final norm + unembedding (optionally tied) + optional multi-token-prediction heads."""

    def __init__(self, cfg, tok_emb: nn.Embedding):
        super().__init__()
        self.cfg = cfg
        self.norm = RMSNorm(cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.proj.weight = tok_emb.weight
        self.mtp = nn.ModuleList([nn.Linear(cfg.d_model, cfg.d_model, bias=False)
                                  for _ in range(getattr(cfg, "mtp_heads", 0))])

    def logits(self, h):
        return self.proj(self.norm(h))

    def forward(self, h, targets=None):
        h = self.norm(h)
        logits = self.proj(h)
        if targets is None:
            return logits, None
        V = logits.size(-1)
        loss = F.cross_entropy(logits.reshape(-1, V).float(), targets.reshape(-1), ignore_index=-100)
        self.main_loss = loss.detach()          # next-token loss only (without MTP terms), for logging
        w = getattr(self.cfg, "mtp_weight", 0.3)
        for k, p in enumerate(self.mtp, start=1):
            if targets.size(1) <= k:
                break
            hk = h[:, :-k] + F.silu(p(h[:, :-k]))
            lk = self.proj(hk)
            loss = loss + w * F.cross_entropy(lk.reshape(-1, V).float(), targets[:, k:].reshape(-1),
                                              ignore_index=-100)
        return logits, loss

### Helpers: gradient checkpointing, weight init, base class

In [166]:
def ckpt_call(fn, enabled, *args):
    return checkpoint(fn, *args, use_reentrant=False) if enabled else fn(*args)


def init_weights(model, depth):
    for m in model.modules():
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
    std = 0.02 / math.sqrt(2 * depth)
    for n, p in model.named_parameters():
        if n.endswith(("o.weight", "down.weight")):
            nn.init.normal_(p, 0.0, std)


class BaseLM(nn.Module):
    """Common interface used by training.py / generation.py:
         forward(x, targets=None, src=None) -> (logits, loss)
         prefill(prompt_ids, max_new)       -> (logits_for_next_token, state)      [generation]
         step(token_ids, state)             -> logits_for_next_token               [generation]
       `eval_offset` = number of leading label positions that are NOT scored (encoder-decoder only)."""
    eval_offset = 0

    def num_params(self, non_embedding=False):
        n = sum(p.numel() for p in self.parameters())
        return n - self.tok.weight.numel() if non_embedding else n

### Linear attention: chunkwise algorithm

In [167]:
def chunk_linear_attn(q, k, v, log_a, chunk=64, return_state=False):
    """q,k: (B,H,T,dk)  v: (B,H,T,dv)  log_a: (B,H,T) with log_a <= 0.
    Equivalent to  S_t = exp(log_a_t) S_{t-1} + k_t^T v_t ;  o_t = q_t S_t.
    With return_state=True also returns the final S (B,H,dk,dv) so generation can continue recurrently."""
    B, H, T, dk = q.shape
    dv = v.shape[-1]
    out_dtype = v.dtype
    q, k, v, log_a = q.float(), k.float(), v.float(), log_a.float()
    pad = (-T) % chunk
    if pad:
        q, k, v = (F.pad(t, (0, 0, 0, pad)) for t in (q, k, v))
        log_a = F.pad(log_a, (0, pad))                 # padded steps: decay 1, k = v = 0  -> state unchanged
    N = (T + pad) // chunk
    q = q.reshape(B, H, N, chunk, dk)
    k = k.reshape(B, H, N, chunk, dk)
    v = v.reshape(B, H, N, chunk, dv)
    b = log_a.reshape(B, H, N, chunk).cumsum(-1)                    # inclusive cumulative log-decay

    causal = torch.tril(torch.ones(chunk, chunk, dtype=torch.bool, device=q.device))
    decay = torch.exp((b.unsqueeze(-1) - b.unsqueeze(-2)).masked_fill(~causal, float("-inf")))
    intra = ((q @ k.transpose(-1, -2)) * decay) @ v                 # (B,H,N,C,dv)

    q_in = q * torch.exp(b).unsqueeze(-1)                           # decay from chunk start to i
    k_out = k * torch.exp(b[..., -1:] - b).unsqueeze(-1)            # decay from j to chunk end
    chunk_decay = torch.exp(b[..., -1])                             # (B,H,N)
    S = q.new_zeros(B, H, dk, dv)
    inter = []
    for n in range(N):
        inter.append(q_in[:, :, n] @ S)
        S = S * chunk_decay[:, :, n, None, None] + k_out[:, :, n].transpose(-1, -2) @ v[:, :, n]
    out = (intra + torch.stack(inter, 2)).reshape(B, H, N * chunk, dv)[:, :, :T].to(out_dtype)
    return (out, S) if return_state else out

### Linear attention layer: parameters and helpers (new: recurrent state for generation)

In [168]:
class LinearState:
    """Recurrent state of one linear-attention layer while generating."""
    def __init__(self):
        self.S = None        # (B,H,dk,dv) float32
        self.conv = None     # (B,3*H*hd,K-1) last inputs of the causal conv


class LinearAttentionCore(nn.Module):
    """baseline  (la_decay="fixed"): RetNet-style multi-scale retention (fixed per-head decay, RoPE,
                 output gate, per-head norm).
       modified  (la_decay="data"):  input-dependent decay gate, short causal conv on q/k/v, L2-normalised
                 q/k, no RoPE (decay supplies position information)."""

    def __init__(self, cfg: Config):
        super().__init__()
        d, H, hd = cfg.d_model, cfg.n_head, cfg.head_dim
        self.H, self.hd, self.chunk = H, hd, cfg.la_chunk
        self.q, self.k, self.v, self.g = (nn.Linear(d, H * hd, bias=False) for _ in range(4))
        self.o = nn.Linear(H * hd, d, bias=False)
        self.onorm = RMSNorm(hd)
        self.l2norm = cfg.la_l2norm
        self.rot_dim = (int(hd * cfg.rope_pct) // 2 * 2) if cfg.la_rope else 0
        self.data_decay = cfg.la_decay == "data"
        if self.data_decay:
            self.a = nn.Linear(d, H)
            with torch.no_grad():                      # multi-scale init: fast ... slow forgetting
                self.a.bias.copy_(torch.linspace(0, 8, H))
        else:
            gamma = 1 - 2.0 ** (-5 - torch.arange(H).float())          # RetNet multi-scale decays
            self.register_buffer("log_gamma", gamma.log().view(1, H, 1), persistent=False)
        self.conv = None
        if cfg.la_conv > 0:
            ch = 3 * H * hd
            self.conv = nn.Conv1d(ch, ch, cfg.la_conv, groups=ch, padding=cfg.la_conv - 1)

    def _prep(self, q, k, rope, B, T):
        """reshape to heads, normalise, rotate"""
        H, hd = self.H, self.hd
        if self.l2norm:
            q, k = F.normalize(q, dim=-1), F.normalize(k, dim=-1)
        else:
            q = q * hd ** -0.5
        if self.rot_dim:
            cos, sin = rope
            q, k = apply_rope(q, cos, sin, self.rot_dim), apply_rope(k, cos, sin, self.rot_dim)
        return q, k

    def _log_a(self, x, B, T):
        if self.data_decay:
            return F.logsigmoid(self.a(x).float()).transpose(1, 2) / 16.0   # (B,H,T), in (-inf,0)
        return self.log_gamma.expand(B, self.H, T)

    def _finish(self, o, x, B, T):
        H, hd = self.H, self.hd
        o = self.onorm(o.transpose(1, 2).to(x.dtype))                      # per-head norm
        o = o * F.silu(self.g(x).view(B, T, H, hd))                        # output gate
        return self.o(o.reshape(B, T, H * hd))

### Linear attention layer: forward pass and one-token recurrent step

In [169]:
class LinearAttention(LinearAttentionCore):
    """forward pass (chunkwise for training / prompt) + one-token recurrent step (generation)."""

    def forward(self, x, rope=None, cache: LinearState = None):
        B, T, _ = x.shape
        H, hd = self.H, self.hd
        q, k, v = self.q(x), self.k(x), self.v(x)
        if cache is not None and T == 1:
            return self._step(x, q, k, v, rope, cache)
        if self.conv is not None:
            cat = torch.cat([q, k, v], -1).transpose(1, 2)                 # (B,ch,T)
            if cache is not None:                                          # prefill: remember conv tail
                K = self.conv.kernel_size[0]
                tail = cat[..., -(K - 1):] if K > 1 else cat[..., :0]
                cache.conv = F.pad(tail, (K - 1 - tail.size(-1), 0))
            qkv = F.silu(self.conv(cat)[..., :T]).transpose(1, 2)
            q, k, v = qkv.split(H * hd, dim=-1)
        q, k, v = (t.reshape(B, T, H, hd).transpose(1, 2) for t in (q, k, v))
        q, k = self._prep(q, k, rope, B, T)
        log_a = self._log_a(x, B, T)
        if cache is not None:
            o, cache.S = chunk_linear_attn(q, k, v, log_a, self.chunk, return_state=True)
        else:
            o = chunk_linear_attn(q, k, v, log_a, self.chunk)              # (B,H,T,hd)
        return self._finish(o, x, B, T)

    def _step(self, x, q, k, v, rope, cache):
        """one-token recurrent update: S <- a*S + k^T v ; o = q S"""
        B, H, hd = x.size(0), self.H, self.hd
        if self.conv is not None:
            cur = torch.cat([q, k, v], -1).transpose(1, 2)                 # (B,ch,1)
            win = torch.cat([cache.conv.to(cur.dtype), cur], -1)           # (B,ch,K)
            cache.conv = win[..., 1:]
            y = (win * self.conv.weight.squeeze(1)).sum(-1, keepdim=True) + self.conv.bias[:, None]
            qkv = F.silu(y).to(x.dtype).transpose(1, 2)
            q, k, v = qkv.split(H * hd, dim=-1)
        q, k, v = (t.reshape(B, 1, H, hd).transpose(1, 2) for t in (q, k, v))
        q, k = self._prep(q, k, rope, B, 1)
        a = self._log_a(x, B, 1).exp().unsqueeze(-1)                       # (B,H,1,1)
        cache.S = cache.S * a + k.float().transpose(-1, -2) @ v.float()
        o = (q.float() @ cache.S).to(v.dtype)                              # (B,H,1,hd)
        return self._finish(o, x, B, 1)

### Transformer block (linear or softmax token mixer)

In [170]:
class Block(nn.Module):
    """Pre-norm residual block with an injected token mixer (linear attention or softmax attention)."""

    def __init__(self, cfg, mixer: nn.Module):
        super().__init__()
        self.n1, self.n2 = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)
        self.mixer, self.mlp = mixer, MLP(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope=None, cache=None):
        h = self.mixer(self.n1(x), rope, cache=cache)
        if isinstance(h, tuple):                      # softmax attention returns (out, v)
            h = h[0]
        x = x + self.drop(h)
        return x + self.drop(self.mlp(self.n2(x)))

### Linear-attention / hybrid model

In [171]:
class DecoderLM(BaseLM):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        assert len(cfg.layer_types) == cfg.n_layer, "layer_types must have n_layer entries"
        self.tok = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.rope = Rotary(int(cfg.head_dim * cfg.rope_pct) // 2 * 2, cfg.max_seq_len, cfg.rope_theta)
        self.blocks = nn.ModuleList([Block(cfg, LinearAttention(cfg) if t == "r" else
                                           Attention(cfg, use_rope=False)) for t in cfg.layer_types])
        self.head = LMHead(cfg, self.tok)
        init_weights(self, cfg.n_layer)

    def hidden(self, idx, caches=None, start=0):
        rope = self.rope(idx.shape[1], start)
        x = self.tok(idx)
        for i, blk in enumerate(self.blocks):
            if caches is None:
                x = ckpt_call(blk, self.cfg.grad_ckpt and self.training, x, rope)
            else:
                x = blk(x, rope, cache=caches[i])
        return x

    def forward(self, idx, targets=None, src=None):
        return self.head(self.hidden(idx), targets)

    # ---- generation: recurrent state for linear layers, KV cache for the softmax layers ----
    def prefill(self, idx, max_new):
        caches = [LinearState() if t == "r" else KVCache(idx.shape[1] + max_new) for t in self.cfg.layer_types]
        h = self.hidden(idx, caches=caches)
        return self.head.logits(h[:, -1]), SimpleNamespace(layers=caches, pos=idx.shape[1])

    def step(self, tok, state):
        h = self.hidden(tok, caches=state.layers, start=state.pos)
        state.pos += 1
        return self.head.logits(h[:, -1])

### Baseline / modified architecture definitions

In [172]:
Model = DecoderLM


BASE = dict(d_model=768, n_layer=12, n_head=12, max_seq_len=2048)


def cfg_baseline(**kw):
    cfg = Config(**{**BASE, "la_decay": "fixed", "la_rope": True, **kw})
    if cfg.layer_types is None:
        cfg.layer_types = ["r"] * cfg.n_layer
    return cfg


def cfg_modified(attn_every=4, **kw):
    cfg = Config(**{**BASE, "n_layer": 16, "la_decay": "data", "la_conv": 4, "la_l2norm": True,
                    "la_rope": False, "n_kv_head": 4, "qk_norm": True, "tie_embeddings": True, **kw})
    if cfg.layer_types is None:
        cfg.layer_types = ["n" if attn_every and (i + 1) % attn_every == 0 else "r"
                           for i in range(cfg.n_layer)]
    return cfg

### Locate and load the Phase-1 checkpoint

In [173]:
def find_ckpt():
    if PHASE1_CKPT and os.path.exists(PHASE1_CKPT):
        return PHASE1_CKPT
    cands = sorted(glob.glob('/kaggle/input/**/*.pt', recursive=True))
    norm = lambda s: s.lower().replace('-', '').replace('_', '')
    hit = [c for c in cands if norm(CKPT_HINT) in norm(c)] or cands
    if not hit:
        raise FileNotFoundError('No Phase-1 checkpoint found: attach it as a Kaggle dataset or set PHASE1_CKPT')
    print('checkpoint candidates:', *hit, sep='\n   ')
    return ([c for c in hit if 'best' in os.path.basename(c)] or [c for c in hit if 'last' in os.path.basename(c)] or hit)[0]


ckpt_path = find_ckpt()
PHASE1 = torch.load(ckpt_path, map_location='cpu', weights_only=False)     # {'model', 'cfg', 'step', 'val_loss'}
D_MODEL = PHASE1['cfg']['d_model']
ARCH_NAME = 'arch3_linear'
assert VOCAB_SIZE <= PHASE1['cfg']['vocab_size'], 'the tokenizer is larger than the checkpoint vocabulary: wrong tokenizer / checkpoint pair'
print('using', ckpt_path, '| Phase-1 step', PHASE1.get('step'), '| val loss', PHASE1.get('val_loss'))

checkpoint candidates:
   /kaggle/input/datasets/kratikasum/model-arch-3/best.pt
using /kaggle/input/datasets/kratikasum/model-arch-3/best.pt | Phase-1 step 600 | val loss 3.583080588053306


### Rebuild the Phase-1 model and read out its hidden states
How the model is used here: causal hybrid: the chunk is processed in parallel (chunkwise linear attention); the **mean** of the token states and the state of the **last** token are the chunk vector.

In [200]:
def load_phase1_backbone(keep_layers=None):
    """Rebuilds the Phase-1 model from the checkpoint (model classes above are unchanged) and drops the LM output layer.
    keep_layers: optional list of layer indices -> a shallower model made of those Phase-1 layers (used for the distillation student)."""
    valid = {f.name for f in fields(Config)}
    cfg = Config(**{k: v for k, v in PHASE1['cfg'].items() if k in valid})
    cfg.grad_ckpt, cfg.dropout = GRAD_CKPT, BACKBONE_DROPOUT
    m = DecoderLM(cfg)
    missing, unexpected = m.load_state_dict(PHASE1['model'], strict=False)
    assert not missing, f'missing keys when loading Phase-1 weights: {missing[:5]}'
    m.head.proj, m.head.mtp = nn.Identity(), nn.ModuleList()                # LM output layer / MTP heads are not needed
    if keep_layers is not None:
        m.cfg.layer_types = [m.cfg.layer_types[i] for i in keep_layers]
        m.blocks = nn.ModuleList([m.blocks[i] for i in keep_layers])
        m.cfg.n_layer = len(keep_layers)
    return m


def get_block_lists(m):
    """The ModuleLists of transformer blocks that the classifier uses (LoRA / LoRMA / adapters are attached to these)."""
    return [m.blocks]


def unused_modules(m):
    """Parts of the Phase-1 model that the classifier never calls (they get no gradient)."""
    return []


def prompt_len(m):
    return getattr(m.tok, 'n_prompt', 0)                                    # > 0 when a soft prompt is attached


def backbone_features(m, ids, mask):
    """(N, T) token ids -> (N, T, d) final-norm hidden states.  The model is causal and padding is on the right, so padding never affects real tokens."""
    p = prompt_len(m)
    if p:
        ids = F.pad(ids, (p, 0), value=PAD_ID)                              # placeholders that PromptedEmbedding replaces by the soft prompt
    return m.head.norm(m.hidden(ids))[:, p:]


FEAT_DIM = 2 * D_MODEL                                                      # chunk vector = [mean | last token]

### Check: parameters of the Phase-1 model

In [201]:
_probe = load_phase1_backbone()
print(f'{ARCH_NAME}: {sum(p.numel() for p in _probe.parameters()) / 1e6:.1f}M parameters loaded from Phase 1 (d_model {D_MODEL})')
del _probe

arch3_linear: 154.3M parameters loaded from Phase 1 (d_model 768)


## 8. Adaptation methods (choose with `METHOD` in the configuration)
The techniques of Assignment 4, re-written for the Phase-1 `nn.Linear` layers.

| `METHOD` | what is trained | idea (Assignment 4) |
|---|---|---|
| `'lorma'` | low-rank matrices `A`, `B` (**compulsory method**) | LoRMA+: `W' = (I + (alpha/r)·B·A)·W0`, computed as `z = W0·x`, `y = z + (alpha/r)·B·(A·z)` (no `d x d` matrix is built) |
| `'lora'` | low-rank matrices `A`, `B` | additive: `W' = W0 + (alpha/r)·B·A` |
| `'adapter'` | a bottleneck MLP after every block | `h' = h + W_up·GELU(W_down·h)`, `W_up = 0` at the start |
| `'soft_prompt'` | learned input vectors | `N_PROMPT` trainable embeddings replace the first positions of every chunk |
| `'full'` | every Phase-1 parameter | full fine-tuning (reference, and the distillation student) |

`B` (LoRA / LoRMA) and `W_up` (adapter) start at zero, so every method starts exactly at the Phase-1 model. The Phase-1 weights are frozen first; the document head is always trained.

### Soft prompt, adapter, LoRA, LoRMA

In [202]:
class SoftPrompt(nn.Module):
    """Assignment 4.1: trainable continuous vectors that replace the first `m` input positions (initialised from real token embeddings)."""
    def __init__(self, init_embeddings):                       # [m, d_model]
        super().__init__()
        self.prompt = nn.Parameter(init_embeddings.detach().clone().float())


class PromptedEmbedding(nn.Module):
    """Wraps the frozen token embedding.  The caller puts `m` placeholder ids in front of every sequence; their embeddings are replaced by the
    learned prompt, so every position-dependent part of the (unchanged) Phase-1 model sees a sequence that is simply m tokens longer."""
    def __init__(self, emb, soft):
        super().__init__()
        self.emb, self.soft = emb, soft
        self.n_prompt = soft.prompt.shape[0]

    @property
    def weight(self):
        return self.emb.weight

    def forward(self, ids):
        x = self.emb(ids)
        prompt = self.soft.prompt.to(x.dtype).unsqueeze(0).expand(x.size(0), -1, -1)
        return torch.cat([prompt, x[:, self.n_prompt:]], dim=1)


class BottleneckAdapter(nn.Module):
    """Assignment 4.2:  h' = h + W_up GELU(W_down h + b_down) + b_up,  with W_up and b_up zero-initialised (identity at step 0)."""
    def __init__(self, hidden_size, bottleneck=16):
        super().__init__()
        self.down, self.up = nn.Linear(hidden_size, bottleneck), nn.Linear(bottleneck, hidden_size)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class AdaptedBlock(nn.Module):
    """A Phase-1 block followed by an adapter (a wrapper module instead of a forward hook, so that nn.DataParallel replicas use their own adapter)."""
    def __init__(self, block, adapter):
        super().__init__()
        self.block, self.adapter = block, adapter

    def forward(self, *args, **kwargs):
        out = self.block(*args, **kwargs)
        if isinstance(out, tuple):                              # some blocks return (hidden, extra)
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)

    def __getattr__(self, name):                                # e.g. `block.window` of the sliding-window architecture
        try:
            return super().__getattr__(name)
        except AttributeError:
            if name == 'block' or name.startswith('__'):
                raise
            return getattr(self.block, name)


class LoRALinear(nn.Module):
    """Assignment 4.3 (LoRA):  y = W0 x + s * B (A x),  A [r, d_in] random, B [d_out, r] zero,  s = alpha / r."""
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


def make_lorma_factors(d_out, rank, generator=None, dtype=torch.float32):
    """Assignment 4.4: base-preserving initialisation, A [r, d_out] ~ U(-1/sqrt(d_out), 1/sqrt(d_out)), B [d_out, r] = 0."""
    bound = 1.0 / math.sqrt(d_out)
    return ((torch.rand(rank, d_out, generator=generator, dtype=torch.float64) * 2 - 1) * bound).to(dtype), torch.zeros(d_out, rank, dtype=dtype)


def lorma_dense(x, W0, factors, scale):
    """Literal LoRMA_+ equation  W' = (s B A + I) W0  (a d_out x d_out matrix is materialised) -> x W'^T.  W0: [d_out, d_in]."""
    A, B = factors
    return x @ ((scale * (B @ A) + torch.eye(W0.shape[0], dtype=B.dtype, device=B.device)) @ W0).T


def lorma_reordered(x, W0, factors, scale):
    """The same function without any d x d product:  h = x W0^T ;  y = h + s (h A^T) B^T."""
    A, B = factors
    h = x @ W0.T
    return h + scale * ((h @ A.T) @ B.T)


class LoRMALinear(nn.Module):
    """LoRMA_+ (multiplicative low-rank adaptation, Assignment 4.4) around a frozen nn.Linear.  Only A and B are trained."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        A, B = make_lorma_factors(base.out_features, r)
        self.A, self.B = nn.Parameter(A), nn.Parameter(B)

    def forward(self, x):
        y = lorma_reordered(x, self.base.weight, (self.A, self.B), self.scale)
        return y if self.base.bias is None else y + self.base.bias


def wrap_linears(module, targets, make):
    """Replaces every nn.Linear called `targets` (q, k, v, o, ...) inside `module` by make(linear); returns how many were replaced."""
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child)); n += 1
        else:
            n += wrap_linears(child, targets, make)
    return n


def prompt_init_embeddings(backbone, n_prompt):
    ids = tokenizer.encode(PROMPT_INIT_TEXT) or [PAD_ID]
    ids = [ids[i % len(ids)] for i in range(n_prompt)]
    return backbone.tok.weight.detach().float().cpu()[torch.tensor(ids)]


def apply_method(backbone, hp):
    """Freezes the Phase-1 weights, then makes a small part trainable according to hp['method'] (the document head is always trained)."""
    method = hp['method']
    for p in backbone.parameters():
        p.requires_grad_(False)
    block_lists = get_block_lists(backbone)
    if method in ('lora', 'lorma'):
        make = (lambda l: LoRALinear(l, hp['r'], hp['alpha'], hp['dropout'])) if method == 'lora' else (lambda l: LoRMALinear(l, hp['r'], hp['alpha']))
        n = sum(wrap_linears(blk, set(hp['targets']), make) for ml in block_lists for blk in ml)
        assert n > 0, f'no linear layer called {hp["targets"]} found'
    elif method == 'adapter':
        for ml in block_lists:
            for i in range(len(ml)):
                ml[i] = AdaptedBlock(ml[i], BottleneckAdapter(D_MODEL, hp['bottleneck']))
    elif method == 'soft_prompt':
        backbone.tok = PromptedEmbedding(backbone.tok, SoftPrompt(prompt_init_embeddings(backbone, hp['n_prompt'])))
    elif method == 'full':
        for p in backbone.parameters():
            p.requires_grad_(True)
    else:
        raise ValueError(f'unknown METHOD {method!r}')
    for mod in unused_modules(backbone):                         # parts of the Phase-1 model that the classifier never calls
        for p in mod.parameters():
            p.requires_grad_(False)

### Sanity checks

In [203]:
# sanity checks of the adaptation code (a few seconds on CPU)
torch.manual_seed(0)
_lin = nn.Linear(24, 20, bias=False).double()
_x = torch.randn(3, 5, 24, dtype=torch.double)
_f = make_lorma_factors(20, 4, dtype=torch.double)
assert torch.allclose(lorma_reordered(_x, _lin.weight, _f, 2.0), _x @ _lin.weight.T), 'LoRMA must start exactly at the Phase-1 layer'
_f = (torch.randn(4, 20, dtype=torch.double) * 0.3, torch.randn(20, 4, dtype=torch.double) * 0.3)
assert torch.allclose(lorma_reordered(_x, _lin.weight, _f, 2.0), lorma_dense(_x, _lin.weight, _f, 2.0)), 'reordered LoRMA != literal equation'
_l = nn.Linear(24, 20, bias=True)
assert torch.allclose(LoRALinear(_l, 4, 8)(_x.float()), _l(_x.float())) and torch.allclose(LoRMALinear(_l, 4, 8)(_x.float()), _l(_x.float()), atol=1e-6)
_h = torch.randn(2, 7, 24); assert torch.equal(BottleneckAdapter(24, 4)(_h), _h)
print('adaptation checks passed: LoRMA / LoRA / adapter all start exactly at the Phase-1 model; LoRMA reordered == dense equation')

adaptation checks passed: LoRMA / LoRA / adapter all start exactly at the Phase-1 model; LoRMA reordered == dense equation


## 9. Document classifier
Phase-1 model (adapted) -> chunk vectors -> hierarchical document head -> outcome.

### Chunk encoder, document head, model builder

In [204]:
class ChunkEncoder(nn.Module):
    """Phase-1 model -> one vector per chunk: [mean of the token states | state of the last real token]."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, ids, mask):
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == 'cuda')):
            h = backbone_features(self.backbone, ids, mask)                           # (N, T, d_token)
        h = h.float()
        m = mask.unsqueeze(-1).float()
        mean = (h * m).sum(1) / m.sum(1).clamp(min=1)
        last = h[torch.arange(h.size(0), device=h.device), (mask.sum(1) - 1).clamp(min=0)]
        return torch.cat([mean, last], -1)                                            # (N, 2 * d_token)


class DocHead(nn.Module):
    """Hierarchical document head: chunk vectors (+ embedding of the chunk's relative position = section-aware) -> pooling -> classifier.
    pool: 'attn' learned attention pooling | 'mean' | 'gru' (BiGRU over the chunk sequence, then mean) | 'last' (last chunk only)."""
    def __init__(self, feat_dim, hidden, n_cls, pool, n_bins):
        super().__init__()
        self.pool = pool
        self.proj = nn.Sequential(nn.LayerNorm(feat_dim), nn.Linear(feat_dim, hidden), nn.GELU())
        self.pos = nn.Embedding(n_bins, hidden)
        if pool == 'attn':
            self.att = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.Tanh(), nn.Linear(hidden // 2, 1))
        elif pool == 'gru':
            self.gru = nn.GRU(hidden, hidden // 2, batch_first=True, bidirectional=True)
        elif pool not in ('mean', 'last'):
            raise ValueError(f'unknown DOC_POOL {pool!r}')
        self.out = nn.Sequential(nn.Dropout(0.1), nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(0.1), nn.Linear(hidden, n_cls))
        self.last_w = None                                                            # attention weights of the last forward pass (analysis)

    def forward(self, feats, batch):
        z = self.proj(feats) + self.pos(batch['pos_bin'])                             # (N, h)
        B, K = batch['n_docs'], batch['max_slots']
        pad = z.new_zeros(B, K, z.size(-1)); pad[batch['doc_idx'], batch['slot']] = z
        valid = torch.zeros(B, K, dtype=torch.bool, device=z.device); valid[batch['doc_idx'], batch['slot']] = True
        n = valid.sum(1)
        if self.pool == 'mean':
            v = (pad * valid.unsqueeze(-1)).sum(1) / n.unsqueeze(-1)
        elif self.pool == 'attn':
            w = torch.softmax(self.att(pad).squeeze(-1).masked_fill(~valid, float('-inf')), 1)
            self.last_w = w.detach()
            v = (w.unsqueeze(-1) * pad).sum(1)
        elif self.pool == 'gru':
            packed = nn.utils.rnn.pack_padded_sequence(pad, n.cpu(), batch_first=True, enforce_sorted=False)
            g, _ = nn.utils.rnn.pad_packed_sequence(self.gru(packed)[0], batch_first=True, total_length=K)
            v = (g * valid.unsqueeze(-1)).sum(1) / n.unsqueeze(-1)
        else:
            v = pad[torch.arange(B, device=z.device), n - 1]
        return self.out(v)


class CJPEModel(nn.Module):
    """Phase-1 model (adapted) -> chunk vectors -> document head -> logits for the 2 outcomes.  forward(batch) -> (logits, chunk_vectors)."""
    def __init__(self, backbone, feat_dim, hp):
        super().__init__()
        self.enc = ChunkEncoder(backbone)
        self.head = DocHead(feat_dim, hp['hidden'], NUM_CLASSES, hp['pool'], N_POS_BINS)
        self.hp = hp

    def use_data_parallel(self):
        """Splits the chunks of a batch over all GPUs (the document head stays on GPU 0).  Not registered as a sub-module on purpose."""
        if N_GPU > 1:
            object.__setattr__(self, '_dp', nn.DataParallel(self.enc))

    def forward(self, batch):
        enc = getattr(self, '_dp', None) or self.enc
        feats = enc(batch['ids'], batch['mask'])
        return self.head(feats, batch), feats


def make_hp(method=None, **kw):
    """All settings that define a model, in one dictionary (also saved with the weights)."""
    hp = dict(method=method or METHOD, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT, targets=list(PEFT_TARGETS), bottleneck=ADAPTER_BOTTLENECK,
              n_prompt=N_PROMPT, pool=DOC_POOL, hidden=HEAD_HIDDEN, keep_layers=None)
    hp.update(kw)
    return hp


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    unused = sum(p.numel() for mod in unused_modules(model.enc.backbone) for p in mod.parameters())
    return dict(total=total, forward_path=total - unused, trainable=sum(p.numel() for p in model.parameters() if p.requires_grad))


def build_model(hp, tag=''):
    """Fresh Phase-1 backbone -> adaptation method -> document head."""
    backbone = load_phase1_backbone(hp['keep_layers'])
    apply_method(backbone, hp)
    model = CJPEModel(backbone, FEAT_DIM, hp).to(device)
    model.use_data_parallel()
    c = count_params(model)
    print(f'{tag}{hp["method"]} | pool={hp["pool"]} | {c["total"] / 1e6:.1f}M parameters ({c["forward_path"] / 1e6:.1f}M used by the classifier), '
          f'{c["trainable"] / 1e6:.2f}M trainable ({100 * c["trainable"] / c["total"]:.2f}%)')
    return model, c

## 10. Training code
Loss = class-weighted cross-entropy with label smoothing. AdamW, warm-up then cosine decay (stretched to the time cap), gradient clipping, fp16 / bf16 autocast, quick validation every `EVAL_EVERY` steps, best weights kept.

### Optimizer, trainable-weight snapshots, prediction, quick validation

In [205]:
def make_optimizer(model, lr, head_lr):
    head = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith('head.')]
    body = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith('head.')]
    groups = [dict(params=head, lr=head_lr, base_lr=head_lr)] + ([dict(params=body, lr=lr, base_lr=lr)] if body else [])
    return torch.optim.AdamW(groups, weight_decay=0.01)


def lr_factor(progress, warmup=0.06):
    if progress < warmup:
        return (progress + 1e-3) / warmup
    return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, (progress - warmup) / (1 - warmup))))


def get_trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}


def load_trainable_state(model, state):
    params = dict(model.named_parameters())
    with torch.no_grad():
        for n, v in state.items():
            params[n].copy_(v)


@torch.no_grad()
def predict_docs(model, docs, mode=None, max_chunks=None, want_attn=False):
    """-> probs (n_docs, 2), info(tokens read, attention weights per document).  Documents of similar size are batched together."""
    mode, max_chunks = mode or SECTION_MODE, max_chunks or MAX_EVAL_CHUNKS
    was_training = model.training
    model.eval()
    sizes = [len(select_chunks(len(d['starts']), mode, max_chunks)) for d in docs]
    order = sorted(range(len(docs)), key=lambda i: sizes[i])
    probs, n_tok, attn, i = np.zeros((len(docs), NUM_CLASSES), np.float32), 0, {}, 0
    while i < len(order):
        ids, c = [], 0
        while i < len(order) and (not ids or c + sizes[order[i]] <= EVAL_CHUNKS_PER_BATCH):
            ids.append(order[i]); c += sizes[order[i]]; i += 1
        batch = make_batch(docs, ids, False, mode=mode, max_chunks=max_chunks)
        logits, _ = model(batch)
        probs[ids] = F.softmax(logits.float(), -1).cpu().numpy()
        n_tok += int(batch['mask'].sum())
        if want_attn and model.head.pool == 'attn':
            w = model.head.last_w.cpu().numpy()
            for b, di in enumerate(ids):
                attn[di] = (batch['rel'][b], w[b, :len(batch['rel'][b])])
    model.train(was_training)
    return probs, dict(tokens=n_tok, attn=attn)


quick_rng = np.random.RandomState(SEED)
quick_dev = [dev_docs[i] for i in sorted(quick_rng.choice(len(dev_docs), min(QUICK_DEV_DOCS, len(dev_docs)), replace=False))]


def quick_eval(model):
    """Selection score during training: macro-F1 on a dev subset (at the best threshold if APPLY_TUNED_THRESHOLD, else at 0.5)."""
    probs, _ = predict_docs(model, quick_dev, max_chunks=QUICK_EVAL_CHUNKS)
    y = np.array([d['y'] for d in quick_dev])
    return tune_threshold(y, probs[:, 1])[1] if APPLY_TUNED_THRESHOLD else binary_scores(y, probs.argmax(1))['macro_f1']

### Training loop

In [206]:
def cjpe_loss(logits, y):
    return F.cross_entropy(logits.float(), y, weight=CLASS_W, label_smoothing=LABEL_SMOOTH)


def run_training(model, tag, epochs, lr, head_lr, teacher=None, kd_loss_fn=None):
    """Trains `model` (and keeps the dev-best weights).  teacher / kd_loss_fn: optional knowledge distillation (used by the KD notebook)."""
    opt = make_optimizer(model, lr, head_lr)
    scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda' and amp_dtype == torch.float16))
    params = [p for g in opt.param_groups for p in g['params']]
    pool_size = len(train_docs) if not EPOCH_DOCS else min(EPOCH_DOCS, len(train_docs))
    docs_per_step = BATCH_DOCS * GRAD_ACCUM
    steps_per_epoch = max(1, pool_size // docs_per_step)
    total_steps = min(steps_per_epoch * epochs, MAX_STEPS or 10 ** 9)
    best, history, bad, step, run_loss, run_n = dict(score=-1.0, step=0, state=None), [], 0, 0, 0.0, 0
    t0, done = time.time(), False
    if teacher is not None:
        teacher.eval()
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    print(f'[{tag}] {steps_per_epoch} optimizer steps per epoch ({docs_per_step} documents each), {total_steps} in total, time cap {MAX_TRAIN_MINUTES} min')
    model.train()
    pbar = tqdm(total=total_steps, desc=tag, dynamic_ncols=True)
    for epoch in range(epochs):
        rng = np.random.RandomState(SEED + epoch)
        perm = rng.permutation(len(train_docs))[:pool_size]
        for s in range(steps_per_epoch):
            progress = max(step / total_steps, (time.time() - t0) / (MAX_TRAIN_MINUTES * 60))
            if progress >= 1.0:
                done = True; break
            for g in opt.param_groups:
                g['lr'] = g['base_lr'] * lr_factor(progress)
            for m in range(GRAD_ACCUM):
                idx = perm[(s * GRAD_ACCUM + m) * BATCH_DOCS:(s * GRAD_ACCUM + m + 1) * BATCH_DOCS]
                batch = make_batch(train_docs, idx, True, rng)
                logits, feats = model(batch)
                if teacher is None:
                    loss = cjpe_loss(logits, batch['y'])
                else:
                    with torch.no_grad():
                        t_logits, t_feats = teacher(batch)
                    loss = kd_loss_fn(logits, feats, t_logits, t_feats, batch['y'])
                scaler.scale(loss / GRAD_ACCUM).backward()
                run_loss += float(loss.detach()) / GRAD_ACCUM
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            step += 1; run_n += 1; pbar.update(1)
            if step % LOG_EVERY == 0:
                pbar.set_postfix(loss=f'{run_loss / run_n:.4f}', lr=f'{opt.param_groups[-1]["lr"]:.1e}')
                history.append(dict(step=step, train_loss=run_loss / run_n)); run_loss, run_n = 0.0, 0
            if step % EVAL_EVERY == 0 or step == total_steps:
                f1 = quick_eval(model); model.train()
                history.append(dict(step=step, quick_dev_macro_f1=f1))
                improved = f1 > best['score']
                if improved:
                    best.update(score=f1, step=step, state=get_trainable_state(model))
                bad = 0 if improved else bad + 1
                pbar.write(f'[{tag}] step {step}/{total_steps} | quick dev macro-F1 {f1:.4f}' + ('  <- best' if improved else '') +
                           f' | {(time.time() - t0) / 60:.1f} min')
                if bad >= PATIENCE:
                    done = True; break
            if step >= total_steps:
                done = True; break
        if done:
            break
    pbar.close()
    if best['state'] is None:                                   # no validation happened (very short run)
        best.update(score=quick_eval(model), step=step, state=get_trainable_state(model))
    load_trainable_state(model, best['state'])
    peak = torch.cuda.max_memory_allocated() / 1e9 if device == 'cuda' else 0.0
    print(f'[{tag}] finished in {(time.time() - t0) / 60:.1f} min | best step {best["step"]} | best quick dev macro-F1 {best["score"]:.4f} | peak GPU memory {peak:.1f} GB')
    return dict(history=history, best_step=best['step'], best_quick_dev_f1=best['score'], seconds=time.time() - t0, peak_mem_gb=peak, steps=step)

## 11. Evaluation and submission helpers

### Validation report, section analysis, attention by position, submission writer

In [207]:
def validate(model, tag, docs=None):
    """Full development report: default threshold, tuned threshold, calibration, error analysis, inference speed."""
    docs = docs if docs is not None else dev_docs
    y = np.array([d['y'] for d in docs])
    t0 = time.time()
    probs, info = predict_docs(model, docs, want_attn=True)
    tok_per_s = info['tokens'] / max(time.time() - t0, 1e-9)
    s_default = binary_scores(y, probs.argmax(1))
    thr, f_tuned = tune_threshold(y, probs[:, 1])
    s_tuned = binary_scores(y, (probs[:, 1] >= thr).astype(int))
    print_scores(s_default, f'[{tag}] threshold 0.5   :')
    print_scores(s_tuned, f'[{tag}] threshold {thr:.2f}  :')
    ece = expected_calibration_error(y, probs)
    print(f'[{tag}] calibration: ECE {ece:.4f} | inference speed {tok_per_s:,.0f} tokens/s')
    errors = error_analysis(docs, probs, thr if APPLY_TUNED_THRESHOLD else 0.5)
    return dict(default=s_default, tuned=s_tuned, threshold=thr, ece=ece, infer_tok_per_s=tok_per_s, errors=errors, probs=probs, attn=info['attn'])


def section_ablation(model, tag, docs=None):
    """Which part of a judgment carries the signal?  The trained model is evaluated while it only reads the first chunk, the last chunk,
    first + last, or the whole judgment (no retraining)."""
    docs = docs if docs is not None else dev_docs
    y = np.array([d['y'] for d in docs]); out = {}
    print(f'[{tag}] what the model reads        macro-F1 @0.5   best-threshold macro-F1   tokens read')
    for mode in ('first', 'last', 'head_tail', 'full'):
        probs, info = predict_docs(model, docs, mode=mode)
        f_default = binary_scores(y, probs.argmax(1))['macro_f1']
        thr, f_best = tune_threshold(y, probs[:, 1])
        out[mode] = dict(macro_f1=f_default, macro_f1_tuned=f_best, threshold=thr, tokens_read=info['tokens'])
        print(f'    {mode:10s}                       {f_default:.4f}               {f_best:.4f}              {info["tokens"]:,}')
    return out


def attention_by_position(attn, bins=5):
    """Average attention weight (document head, 'attn' pooling) per relative position in the judgment, relative to a uniform weight."""
    if not attn:
        return None
    acc, cnt = np.zeros(bins), np.zeros(bins)
    for rel, w in attn.values():
        if len(w) < 2:
            continue
        b = np.minimum((rel * bins).astype(int), bins - 1)
        np.add.at(acc, b, w * len(w)); np.add.at(cnt, b, 1)
    share = acc / np.maximum(cnt, 1)
    print('attention weight by position in the judgment (1.0 = uniform):', ' | '.join(f'{100 * i // bins}-{100 * (i + 1) // bins}%: {share[i]:.2f}' for i in range(bins)))
    return share.tolist()


def write_submission(test_docs, probs, thr, out_dir):
    """prediction.json = [{"id": ..., "pred": 0/1}, ...] in the order of test.json, plus submission.zip containing it."""
    pred = (probs[:, 1] >= thr).astype(int)
    rows = [dict(id=d['id'], pred=int(p)) for d, p in zip(test_docs, pred)]
    assert len(rows) == len(test_docs) and all(r['pred'] in (0, 1) and isinstance(r['pred'], int) for r in rows)
    path = os.path.join(out_dir, 'prediction.json')
    json.dump(rows, open(path, 'w', encoding='utf-8'), indent=1)
    with zipfile.ZipFile(os.path.join(out_dir, 'submission.zip'), 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname='prediction.json')
    print(f'wrote {path} and submission.zip | {len(rows)} predictions | predicted label counts {dict(Counter(int(p) for p in pred))} | threshold {thr:.2f}')


def summary_table(rows):
    """rows = [(name, param counts, train result, validation result)] -> compact comparison used for the report"""
    print(f'{"experiment":26s}{"params":>9s}{"fwd path":>10s}{"trainable":>11s}{"macro-F1":>10s}{"tuned F1":>10s}{"macro-P":>9s}{"macro-R":>9s}{"tok/s":>9s}{"min":>7s}{"GPU GB":>8s}')
    for name, c, tr, v in rows:
        s = v['tuned'] if APPLY_TUNED_THRESHOLD else v['default']
        print(f'{name:26s}{c["total"] / 1e6:8.1f}M{c["forward_path"] / 1e6:9.1f}M{c["trainable"] / 1e6:10.2f}M{v["default"]["macro_f1"]:10.4f}{v["tuned"]["macro_f1"]:10.4f}'
              f'{s["macro_p"]:9.4f}{s["macro_r"]:9.4f}{v["infer_tok_per_s"]:9.0f}{tr["seconds"] / 60:7.1f}{tr["peak_mem_gb"]:8.1f}')

## 12. Run the experiment

### 12a. Build the model

In [209]:
HP = make_hp()
model, param_counts = build_model(HP)

adapter | pool=attn | 156.4M parameters (156.4M used by the classifier), 2.08M trainable (1.33%)


In [210]:
# --- Fallback 1: validate less often, on fewer documents ---
EVAL_EVERY = 400
QUICK_DEV_DOCS = 120
QUICK_EVAL_CHUNKS = 6
quick_dev = [dev_docs[i] for i in sorted(np.random.RandomState(SEED).choice(len(dev_docs), min(QUICK_DEV_DOCS, len(dev_docs)), replace=False))]

# --- Fallback 2: single GPU (only use if RAM still climbs) ---
model.__dict__.pop('_dp', None)

DataParallel(
  (module): ChunkEncoder(
    (backbone): DecoderLM(
      (tok): Embedding(48000, 768)
      (rope): Rotary()
      (blocks): ModuleList(
        (0-2): 3 x AdaptedBlock(
          (block): Block(
            (n1): RMSNorm()
            (n2): RMSNorm()
            (mixer): LinearAttention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (g): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (onorm): RMSNorm()
              (a): Linear(in_features=768, out_features=12, bias=True)
              (conv): Conv1d(2304, 2304, kernel_size=(4,), stride=(1,), padding=(3,), groups=2304)
            )
            (mlp): MLP(
              (gate): Linear(in_features=768, out_features=2048, bias=False)
           

### 12b. Train

In [ ]:
train_res = run_training(model, RUN_NAME, EPOCHS, LR, HEAD_LR)

[p4_arch3_adapter] 1500 optimizer steps per epoch (8 documents each), 3000 in total, time cap 300 min


p4_arch3_adapter:   0%|          | 0/3000 [00:00<?, ?it/s]

### 12c. Validate: scores, threshold, calibration, false positives / negatives

In [ ]:
val_res = validate(model, RUN_NAME)

### 12d. Which part of the judgment matters? (reading only the first / last / first + last / all chunks)

In [ ]:
section_res = section_ablation(model, RUN_NAME)
attn_res = attention_by_position(val_res['attn'])

### 12e. Predict the test cases and write the submission

In [ ]:
test_probs, _ = predict_docs(model, test_docs)
THRESHOLD = val_res['threshold'] if APPLY_TUNED_THRESHOLD else 0.5
write_submission(test_docs, test_probs, THRESHOLD, OUTPUT_DIR)

### 12f. Summary: size, memory, speed, scores

In [ ]:
summary_table([(f'{METHOD} / {SECTION_MODE}', param_counts, train_res, val_res)])

## 13. Save weights and results

In [ ]:
np.save(os.path.join(OUTPUT_DIR, 'dev_probs.npy'), val_res['probs']); np.save(os.path.join(OUTPUT_DIR, 'test_probs.npy'), test_probs)
json.dump(dict(dev_ids=[d['id'] for d in dev_docs], test_ids=[d['id'] for d in test_docs]), open(os.path.join(OUTPUT_DIR, 'ids.json'), 'w'))
torch.save(dict(trainable=get_trainable_state(model), hp=HP, feat_dim=FEAT_DIM, chunk_len=CHUNK_LEN, section_mode=SECTION_MODE, threshold=THRESHOLD,
                phase1_ckpt=ckpt_path), os.path.join(OUTPUT_DIR, f'{RUN_NAME}_trainable.pt'))
drop = lambda v: {k: x for k, x in v.items() if k not in ('probs', 'attn')}
json.dump(dict(run=RUN_NAME, arch=ARCH_NAME, hp=HP, params=param_counts, train=train_res, validation=drop(val_res), section_ablation=section_res,
               attention_by_position=attn_res, threshold=THRESHOLD, class_counts=counts.tolist(), class_weights=cw.tolist()),
          open(os.path.join(OUTPUT_DIR, 'results.json'), 'w'), indent=1, default=str)
print('saved:', sorted(os.listdir(OUTPUT_DIR)))